In [0]:
%run /Workspace/Users/pramotha@systechusa.com/qa-validation-framework/config/validation_queries_and_logs

In [0]:
import pandas as pd

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS project_training_amh.default.qa_validation_results (
    check_name STRING,
    status STRING,
    src_count INT,
    tgt_count INT,
    mismatch_count INT,
    duplicate_count INT,
    null_count INT,
    remarks STRING,
    run_id STRING,
    timestamp TIMESTAMP
)
""")    

In [0]:
src_table = "project_training_amh.default.employee_src_qa"
tgt_table = "project_training_amh.default.employee_tgt_qa"

In [0]:
results = []

# Row Count
results.append(row_count_check(spark, src_table, tgt_table))

# Null Check
# results.append(null_check(spark, src_table))

# Duplicate Check
results.append(duplicate_check(spark, tgt_table, "emp_id"))

# GroupBy Check
results.append(groupby_check(spark, src_table, tgt_table, ["dept_id"]))

# Referential Check
results.append(referential_check(spark, tgt_table, src_table, "emp_id"))

In [0]:
display(pd.DataFrame(results))

In [0]:
log_results(spark, results)

In [0]:
%sql
select * from project_training_amh.default.qa_validation_results

In [0]:
%skip
data = [
(1,"Arun",101,50000),
(2,"Bala",102,60000),
(3,"Chitra",101,55000),
(4,"Deepak",103,70000),
(5,"Esha",104,65000),
(6,"Farhan",102,48000),
(7,"Gita",101,52000),
(8,"Hari",103,72000),
(9,"Isha",104,58000),
(10,"John",102,61000),
(11,"Kiran",101,53000),
(12,"Latha",104,67000),
(13,"Manoj",103,75000),
(14,"Nisha",102,62000),
(15,"Omkar",101,54000),
(16,"Pooja",104,69000),
(17,"Qadir",103,71000),
(18,"Ravi",102,60000),
(19,"Sneha",101,56000),
(20,"Tarun",104,68000)
]

cols = ["emp_id","emp_name","dept_id","salary"]

spark.createDataFrame(data, cols) \
    .write.mode("overwrite") \
    .saveAsTable("project_training_amh.default.employee_src_qa")

In [0]:
%skip
data = [
(1,"Arun",101,50000),
(2,"Bala",102,60000),
(3,"Chitra",101,55000),
(4,"Deepak",103,70000),
(5,"Esha",104,None),        # NULL issue
(6,"Farhan",102,48000),
(7,"Gita",101,52000),
(8,"Hari",103,72000),
(9,"Isha",104,58000),
(10,"John",102,61000),
(11,"Kiran",101,53000),
(12,"Latha",104,67000),
(13,"Manoj",103,75000),
(14,"Nisha",102,62000),
(15,"Omkar",101,99999),     # Mismatch
(16,"Pooja",104,69000),
(17,"Qadir",103,71000),
(18,"Ravi",102,60000),
(19,"Sneha",101,56000),
(19,"Sneha",101,56000)      # Duplicate
# Missing emp_id = 20
]

cols = ["emp_id","emp_name","dept_id","salary"]

spark.createDataFrame(data, cols) \
    .write.mode("overwrite") \
    .saveAsTable("project_training_amh.default.employee_tgt_qa")